# Continued pretraining on geo-triples-japan

One pass of continued pretraining on a corpus of spatial facts about Japan,
with the measurement on both sides of it. Runs top to bottom on a Colab A100
and needs nothing from disk.

The corpus is the `cpt` subset of
[`yuiseki/geo-triples-japan`](https://huggingface.co/datasets/yuiseki/geo-triples-japan),
filtered to the Japanese sentences about the administrative hierarchy. Every one states a spatial fact computed from frozen,
openly licensed geometry by a Docker-pinned GeoSPARQL endpoint, and every step
from a DE-9IM matrix to a predicate is proved by a Lean development. Nothing
in it was written by a model.

```
南伊勢町は三重県に含まれる。
柏市は白井市と接している。
南伊勢町は三重県に含まれる。三重県は日本に含まれる。したがって南伊勢町は日本に含まれる。
```

## What this run is

The narrowest question worth asking first: **can a 0.6B model hold the
administrative hierarchy of Japan?** 1,740 municipalities, which prefecture
each is in, which of them border each other, and 47 prefectures in one
country. 39,126 Japanese sentences, 1,180,982 characters, 996,830 tokens. One pass
is 60 steps, so this runs in minutes and can afford ten passes.

That is two orders of magnitude smaller than the same dataset's places, and
deliberately. A 0.6B model memorising every shrine and school in Japan is not
the first thing to find out. Set `TOPIC = "place"` for that one, and expect
it to be a different experiment rather than a bigger version of this one.

## Why these defaults

A first run on the Tokyo dataset made the model worse, and the way it failed
decided the settings here.

It trained on all three forms, so four fifths of its budget went on N-Triples
IRIs and only 8% on Japanese sentences. Japanese `place-in-ward` fell from
22.5% to 11.7%, English rose, and the control corpus loss rose by 0.48, which
is what forgetting looks like before it reaches a benchmark. The answers had
collapsed onto one ward: everything was Koto in English and 中野区 in
Japanese, with invented wards like 西武区 among them. The model learnt the
shape of an answer and lost the content.

So: one form, one topic, and a learning rate a third lower on a corpus forty
times smaller. One variable at a time.

`REPLAY` is the other lever and it is off. If the control loss rises again,
set it to 0.2 and run once more; mixing general text back in is the ordinary
remedy, and it is left off here so that this run measures one thing.

## What this measures, and what the first run got wrong

The first run of this notebook failed, and one of the two reasons was the
measurement rather than the training.

It asked every question in a question-and-answer shape the corpus never uses.
The corpus says 南伊勢町は三重県に含まれる。 and the probe asked
Q: 南伊勢町はどの都道府県にありますか。 A: . A base model continued on
declarative sentences gets better at declarative sentences and worse at
following a few-shot pattern, so it can lose that score without losing the
fact. Its held-out loss fell by 0.60 while its answers collapsed onto one
prefecture, 福井県 for everything.

So there are two ways of asking now and they are reported apart.

**cloze** opens the corpus's own sentence: 松山市は . Whether the fact went in.

**qa** asks the question with three worked examples. Whether the fact can be
reached by being asked.

A rise in cloze with no rise in qa is the expected shape for continued
pretraining on declarative text, and it says instruction tuning is the next
step rather than that the corpus failed.

The other reason the first run failed was the control loss, which rose by
0.48. `REPLAY` is 0.2 now: a fifth of each batch is drawn from Japanese
Wikipedia instead. Half the control corpus is used for that and the other
half is never trained on, so the measurement still means something.

**Two controls on top of that.** 39% of the place questions carry the answer
inside the subject's own name, and the dataset marks them, so every number is
reported with and without. And one place in ten is held out of the corpus
entirely: the eval half cannot be recalled and is not expected to move. If it
moves with the train half, the model has learnt to answer with a plausible
municipality rather than the right one, which is exactly what the first run
did.

What the first run measured, asked the one way it knew, on
`municipality-in-prefecture` in Japanese: 6.7% before and 0.8% after with the
leak, 3.3% before and 6.7% after without it. A 35B model gets 90.0% and
86.7%. Chance is 2.1%.

The control corpus is Japanese Wikipedia, never trained on. Training Japanese
templates while losing Japanese prose is the specific risk here, so the
control is in the same language this time.

Ten passes over the administrative corpus is a few minutes of A100 time. The
probes cost more than the training does at this size. Under a compute unit
for the whole notebook.

## 1. The machine, and what to install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "transformers>=4.57" datasets pyarrow

## 2. Configuration

The base model rather than the instruction-tuned one. Continued pretraining
undoes instruction tuning before it teaches anything, so the order that makes
sense is to continue from the base and instruction-tune afterwards.

Qwen rather than gemma, for two reasons. `google/gemma-3-270m` is a gated
repository and a notebook that needs a licence acceptance and a token does not
run as it stands. And four fifths of this corpus is N-Triples, where Qwen's
tokenizer is the better one: 2.71 characters per token against gemma's 2.23,
which is 31 million tokens rather than 39 for the same text. Japanese goes the
other way, 1.39 against 1.66, and the N-Triples majority decides it.

To use gemma anyway, set `MODEL` and add a cell with `huggingface_hub.login()`
before section 4.

In [ ]:
DATASET   = "yuiseki/geo-triples-japan"
REVISION  = None          # pin a commit sha here to fix what is measured
MODEL     = "Qwen/Qwen3-0.6B-Base"
FORMS     = "ja"          # "ja", "en", "ntriples", or None for all three
TOPIC     = "admin"       # "admin", "place", or None for both
TRAIN_ONLY = True         # drop the rows that name a held-out place

# Where the memory goes is the logits, not the model: batch x block x 151,669
# vocabulary entries, upcast to fp32 for the loss. The first step prints the
# peak. If it runs out, halve BATCH before touching anything else; gradient
# accumulation makes that free, because 8 x 2 and 4 x 4 see the same tokens.
BLOCK       = 1024
BATCH       = 8
GRAD_ACCUM  = 2
LR          = 2e-5        # 3e-5 collapsed a run 40 times this size
EPOCHS      = 10.0        # admin is 996,830 tokens; one pass is 60 steps
EVAL_EVERY  = 100
EVAL_ITERS  = 20
EVAL_BATCH  = 1           # eval builds the logits; training with liger does not
SEED        = 7
PROBE_N     = 120         # questions per level per language
CONTROL      = "20260901.ja"   # Wikipedia, never trained on
CONTROL_DOCS = 200
REPLAY       = 0.2        # fraction of control text mixed into training
OUT         = "runs/geo"

# For a smoke run before spending an hour: a few documents and a few steps
# take the same path as the real thing. 0 means the whole corpus.
LIMIT     = 0
MAX_STEPS = 0

## 3. The pipeline

A copy of `src/corpus.py`, `src/train.py` and `src/hierarchy_probe.py` from
the repository this notebook came from, flattened into one cell so the
notebook needs no checkout. Three things in it are load-bearing and easy to
get wrong, so they are commented where they happen: the labels are not
shifted by the caller, the held-out tail needs room for a window to start in,
and the probe is sampled per level.

In [ ]:
import collections, json, math, os, random, time
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, \
    get_cosine_schedule_with_warmup

DTYPE = np.uint32   # 151,669 vocabulary entries for Qwen, 262,144 for gemma;
                    # neither fits in 16 bits


def build_corpus(out, dataset, config, model, where=None, limit=0,
                 text_field="text", revision=None):
    """A published dataset -> one flat array of token ids on disk.

    Documents are separated by the end-of-text id. Without it a window that
    straddles two documents reads as one run-on document, and the model learns
    to continue from one fact into an unrelated one.

    The limit counts documents kept, not documents read: the cpt table is
    sorted by form, so every ja row sits after 202,116 ntriples rows and
    counting rows read would make a small limit return nothing.
    """
    tok = AutoTokenizer.from_pretrained(model)
    eos = tok.eos_token_id
    if eos is None:
        raise SystemExit(f"{model} has no end-of-text token; there is nothing "
                         "to separate documents with")
    ds = load_dataset(dataset, config, split="train", revision=revision)
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    total, kept, buf = 0, 0, []
    pending = []

    def flush(f):
        nonlocal total, buf
        if buf:
            np.asarray(buf, dtype=DTYPE).tofile(f)
            total += len(buf)
            buf = []

    with open(out, "wb") as f:
        for row in ds:
            if where and any(str(row.get(k)) != v for k, v in where.items()):
                continue
            text = row.get(text_field)
            if not text:
                continue
            pending.append(text)
            kept += 1
            if len(pending) >= 256:
                for ids in tok(pending, add_special_tokens=False)["input_ids"]:
                    buf.extend(ids + [eos])
                pending = []
                if len(buf) >= 1_000_000:
                    flush(f)
            if limit and kept >= limit:
                break
        if pending:
            for ids in tok(pending, add_special_tokens=False)["input_ids"]:
                buf.extend(ids + [eos])
        flush(f)
    print(f"{total:,} tokens from {kept:,} documents -> {out}")
    return total


def split_holdout(n, holdout, block):
    """Where training stops and the held-out tail begins.

    A tail rather than a random sample, because windows overlap and a randomly
    chosen evaluation window shares tokens with the training windows either
    side of it.

    The floor is block + 2, not block + 1: a tail of exactly block + 1 holds
    one window and no choice of where to start it, and a short corpus then
    dies after the model has loaded.
    """
    return max(0, n - max(block + 2, int(n * holdout)))


def batch(tokens, lo, hi, batch_size, block, rng, device):
    """One batch of windows.

    A window, not a pair. Transformers shifts labels itself, so the caller
    passes the same window as input_ids and as labels. Handing it a window
    already shifted by one asks for the token after next, and the loss sits at
    chance while the run completes normally.
    """
    top = hi - block - 1
    starts = rng.integers(lo, top, size=batch_size)
    x = np.stack([tokens[s:s + block] for s in starts]).astype(np.int64)
    return torch.from_numpy(x).to(device, non_blocking=True)


@torch.no_grad()
def evaluate(model, tokens, lo, hi, block, iters, device, seed, bs=1):
    rng = np.random.default_rng(seed)
    model.eval()
    total = 0.0
    for _ in range(iters):
        x = batch(tokens, lo, hi, bs, block, rng, device)
        total += float(model(input_ids=x, labels=x).loss)
    model.train()
    return total / iters

In [ ]:
CANDIDATES = {"place-in-ward": 23, "state-in-country": 258, "ward-in-state": 47,
              "place-in-municipality": 1740, "municipality-in-prefecture": 47}
SHOTS = 3

QUESTION = {
    ("place-in-ward", "en"):    "Which ward of Tokyo is {child} in?",
    ("place-in-ward", "ja"):    "{child}は東京都のどの区にありますか。",
    ("place-in-municipality", "en"): "Which municipality of Japan is {child} in?",
    ("place-in-municipality", "ja"): "{child}はどの市区町村にありますか。",
    ("municipality-in-prefecture", "en"): "Which prefecture is {child} in?",
    ("municipality-in-prefecture", "ja"): "{child}はどの都道府県にありますか。",
    ("ward-in-state", "en"):    "Which prefecture is {child} in?",
    ("ward-in-state", "ja"):    "{child}はどの都道府県にありますか。",
    ("state-in-country", "en"): "Which country is {child} in?",
    ("state-in-country", "ja"): "{child}はどの国にありますか。",
}

# Q and A on their own lines, and the last one left open. A 0.6B base model
# needs the format: with the examples merely separated by a blank line it
# produced Chinese trivia, or repeated the first example's question back.
#
# The open turn ends at the colon with no space after it. A trailing space is
# its own token and the first token of the answer then arrives without its
# leading one: 文京区 came back as 京区, and every answer was wrong for a
# reason that had nothing to do with the model.
TURN = "Q: {q}\nA: {a}"
OPEN = "Q: {q}\nA:"

# The other way to ask, and for a model continued on declarative sentences it
# is the fair one. The corpus says "南伊勢町は三重県に含まれる。" and the
# question above asks for the same fact in a shape the corpus never uses. A
# first run of this notebook lost 6 points of question-and-answer score while
# its held-out loss fell by 0.60, which is what losing the format rather than
# the fact looks like. Both are measured now and reported apart.
CLOZE = "{child}は"


def bare(name):
    """愛媛県 -> 愛媛, so an answer in either form counts.

    Not 区: 港区 and 港 are not interchangeable the way 愛媛県 and 愛媛 are,
    and dropping it would let 北区 match 北千住.
    """
    return name[:-1] if name and name[-1] in "県府都道" else name


def correct(answer, expected, mode="qa"):
    """Only the first line counts, and it must not be another question.

    The model is continuing a list of question-and-answer pairs, so after its
    answer it writes the next question, and that question names a place.
    """
    first = (answer or "").strip().split("\n")[0]
    if mode == "cloze":
        # Continuing a sentence, so the answer sits inside the clause rather
        # than alone. One line still: the next sentence names other places.
        return bool(first) and bare(expected) in first
    if first.startswith("Q:") or first.startswith("Q："):
        return False
    return bool(first) and bare(expected) in first


def few_shot(rows, lang, k=SHOTS):
    """Worked examples, spread across the level, no two with the same answer.

    A base model asked cold continues the question instead of answering it.
    The first three by id at the country level were Aruba and two provinces of
    Afghanistan, and the model then answered アフガニスタン to everything, so
    they are spread rather than taken from the front. They come out of the
    pool, so a demonstration is never also scored, and the same three are used
    before and after training.
    """
    by_level = collections.defaultdict(list)
    for r in rows:
        by_level[r["level"]].append(r)
    shots, rest = {}, []
    for level in sorted(by_level):
        group = sorted(by_level[level], key=lambda r: r["child_id"])
        order = list(range(len(group)))
        random.Random(11).shuffle(order)
        picked, parents = [], set()
        for i in order:
            if len(picked) >= k:
                break
            if group[i]["parent_id"] in parents:
                continue
            parents.add(group[i]["parent_id"])
            picked.append(i)
        used = [group[i] for i in sorted(picked)]
        group = [r for i, r in enumerate(group) if i not in set(picked)]
        shots[level] = "".join(
            TURN.format(q=QUESTION[(level, lang)].format(
                child=u["child_ja"] if lang == "ja" else u["child_en"]),
                a=u["parent_ja"] if lang == "ja" else u["parent_en"]) + "\n\n"
            for u in used)
        rest.extend(group)
    return shots, rest


def probe_rows(n, lang, seed=3, dataset=DATASET, revision=None,
               exclude_leaks=False, split=None):
    """n questions from each level, not n from the whole set.

    There are 66,541 place questions and 1,484 municipality ones, so a sample
    drawn over both is a sample of the first. Rows without a label in this
    language are dropped: most places carry no name:en, and asking which
    municipality None is in scores the model on a question nobody could
    answer.

    exclude_leaks drops the questions whose subject already names the answer.
    39% of the place questions do, and answering those is reading rather than
    recall.
    """
    rows = [dict(r) for r in load_dataset(dataset, "probe", split="train",
                                          revision=revision)]
    rows = [r for r in rows
            if r.get("child_" + lang) and r.get("parent_" + lang)]
    if exclude_leaks:
        rows = [r for r in rows if not r.get("answer_in_child_" + lang)]
    if split:
        rows = [r for r in rows if r.get("split") == split]
    rows.sort(key=lambda r: (r["level"], r["child_id"]))
    shots, rows = few_shot(rows, lang)
    by_level = collections.defaultdict(list)
    for r in rows:
        by_level[r["level"]].append(r)
    out = []
    for level in sorted(by_level):
        group = by_level[level]
        random.Random(seed).shuffle(group)
        out.extend(group[:n] if n else group)
    out.sort(key=lambda r: (r["level"], r["child_id"]))
    return shots, out


def ask(model, tok, shots, rows, lang, device, chat=False, mode="qa"):
    """The questions, as a completion.

    Asked as plain text unless chat is on. Qwen3-0.6B-Base ships a chat
    template although it is a base model, and wrapping a few-shot block in
    ChatML makes it answer by continuing the list of questions: every score
    was zero, and none of it was about the model's geography.
    """
    templated = chat and getattr(tok, "chat_template", None)
    model.eval()
    out = []
    for i, row in enumerate(rows):
        child = row["child_ja"] if lang == "ja" else row["child_en"]
        if mode == "cloze":
            text = CLOZE.format(child=child)
        else:
            q = QUESTION[(row["level"], lang)].format(child=child)
            text = shots.get(row["level"], "") + OPEN.format(q=q)
        if templated:
            text = tok.apply_chat_template([{"role": "user", "content": text}],
                                           tokenize=False,
                                           add_generation_prompt=True)
        ids = tok(text, return_tensors="pt").to(device)
        with torch.no_grad():
            gen = model.generate(**ids, max_new_tokens=16, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        out.append((row, lang, tok.decode(gen[0][ids["input_ids"].shape[1]:],
                                          skip_special_tokens=True)))
        if (i + 1) % 100 == 0:
            print(f"    {i + 1}/{len(rows)}", flush=True)
    model.train()
    return out


def score(label, answers, mode="qa"):
    per = collections.defaultdict(lambda: [0, 0])
    wrong = []
    for row, lang, got in answers:
        expected = row["parent_ja"] if lang == "ja" else row["parent_en"]
        ok = correct(got, expected, mode)
        per[row["level"]][1] += 1
        per[row["level"]][0] += ok
        if not ok and len(wrong) < 6:
            wrong.append((row, lang, got, expected))
    print(f"\n{label}")
    for level, (hit, n) in sorted(per.items()):
        print(f"  {level:18} {hit:5}/{n:<5} {hit / n:6.1%}   "
              f"(chance {1 / CANDIDATES[level]:.1%})")
    for row, lang, got, expected in wrong:
        child = row["child_ja"] if lang == "ja" else row["child_en"]
        print(f"    {child} -> {expected} | "
              f"{got.strip().split(chr(10))[0][:46]}")
    return {level: {"correct": h, "n": n, "accuracy": h / n,
                    "chance": 1 / CANDIDATES[level]}
            for level, (h, n) in per.items()}

## 4. The corpus, and the control it is measured against

In [ ]:
os.makedirs("data", exist_ok=True)
where = {}
if FORMS:
    where["form"] = FORMS
if TOPIC:
    where["topic"] = TOPIC
if TRAIN_ONLY:
    # The held-out places, dropped here rather than filtered later, so that
    # nothing downstream has to remember to.
    where["holdout"] = "False"
n_corpus = build_corpus("data/geo.bin", DATASET, "cpt", MODEL,
                        where=where or None, limit=LIMIT, revision=REVISION)
n_control = build_corpus("data/control.bin", "yuiseki/wikipedia-geotagged",
                         CONTROL, MODEL, limit=CONTROL_DOCS)

## 5. Before

Two languages, three levels, sampled per level. Takes a few minutes: the
questions are answered one at a time so that the decode is the same before and
after.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16 if device == "cuda" else torch.float32).to(device)

# Which conditions to ask under. Every combination is 8 runs of PROBE_N
# questions on each side of training, which is most of the probe's cost, so
# the defaults ask what this run can answer: Japanese, both halves of the
# split, with and without the questions that name their own answer.
#
# The eval half is expected to sit still. It is the control that says a rise
# in the train half is the facts going in rather than the model learning to
# answer with a plausible municipality.
CONDITIONS = [(mode, leaks, split)
              for mode in ("cloze", "qa")
              for leaks in (True, False)
              for split in ("train", "eval")]


def run_probe(model, tok, label):
    out = {}
    for mode, leaks, split in CONDITIONS:
        shots, rows = probe_rows(PROBE_N, "ja", revision=REVISION,
                                 exclude_leaks=not leaks, split=split)
        if not rows:
            continue
        tag = f"{mode} {split}" + ("" if leaks else " no-leak")
        print(f"{len(rows)} questions, {tag}")
        out[tag] = score(f"{label} [{tag}]",
                         ask(model, tok, shots, rows, "ja", device, mode=mode),
                         mode)
    return out


before = run_probe(model, tok, f"{MODEL} before")

## 6. The run

In [ ]:
torch.manual_seed(SEED)
tokens = np.memmap("data/geo.bin", dtype=DTYPE, mode="r")
control = np.memmap("data/control.bin", dtype=DTYPE, mode="r")
cut = split_holdout(len(tokens), 0.005, BLOCK)

# With replay on, half the control corpus is trained on and the other half is
# not. A corpus that is mixed into training is no longer a control, and
# measuring forgetting against text the model has just seen would report that
# there is none.
replay_cut = len(control) // 2 if REPLAY else 0
print(f"replay {REPLAY:.0%}"
      + (f", drawn from the first {replay_cut:,} control tokens" if REPLAY
         else "; the control is never trained on"))

per_step = BATCH * GRAD_ACCUM * BLOCK
steps = MAX_STEPS or max(1, int(EPOCHS * cut / per_step))
print(f"{len(tokens):,} tokens, training on {cut:,}, "
      f"holding out {len(tokens) - cut:,}")
print(f"{steps:,} steps of {per_step:,} tokens "
      f"({BATCH} x {GRAD_ACCUM} x {BLOCK})")

model.gradient_checkpointing_enable()
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01,
                        betas=(0.9, 0.95))
sched = get_cosine_schedule_with_warmup(opt, min(100, steps // 10 + 1), steps)

os.makedirs(OUT, exist_ok=True)
log = []
rng = np.random.default_rng(SEED)
began = time.time()


def checkpoint(step):
    row = {"step": step,
           "held_out": evaluate(model, tokens, cut, len(tokens), BLOCK,
                                EVAL_ITERS, device, SEED, EVAL_BATCH),
           "control": evaluate(model, control, replay_cut, len(control),
                               BLOCK, EVAL_ITERS, device, SEED, EVAL_BATCH),
           "elapsed_s": round(time.time() - began, 1)}
    log.append(row)
    print(f"  step {step:>6}  held_out {row['held_out']:.4f}  "
          f"control {row['control']:.4f}", flush=True)
    return row


first = checkpoint(0)
chance = math.log(model.config.vocab_size)
if first["held_out"] > chance - 1.0:
    # A pretrained model reading text sits far below a uniform draw. Sitting
    # near it means the model is being asked the wrong question, which is what
    # a shifting mistake looks like from the outside.
    raise SystemExit(
        f"the starting loss is {first['held_out']:.2f} against {chance:.2f} "
        "for a uniform draw. Something is wrong before any training has "
        "happened: the wrong tokenizer for this corpus, or labels shifted "
        "twice. Fix it rather than watching this number come down.")

for step in range(1, steps + 1):
    opt.zero_grad(set_to_none=True)
    for _ in range(GRAD_ACCUM):
        if REPLAY and rng.random() < REPLAY:
            x = batch(control, 0, replay_cut, BATCH, BLOCK, rng, device)
        else:
            x = batch(tokens, 0, cut, BATCH, BLOCK, rng, device)
        (model(input_ids=x, labels=x).loss / GRAD_ACCUM).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    if step == 1 and device == "cuda":
        print(f"  peak VRAM {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB")
    if step % EVAL_EVERY == 0 or step == steps:
        checkpoint(step)

seen = steps * per_step
print(f"{seen:,} tokens in {time.time() - began:.0f}s")
model.save_pretrained(OUT)
tok.save_pretrained(OUT)

## 7. After

In [ ]:
after = run_probe(model, tok, f"{MODEL} after")

print("\n=== recall, on places the corpus states outright")
for tag in sorted(before):
    for level in sorted(before[tag]):
        b, a = before[tag][level], after[tag][level]
        print(f"  [{tag:10}] {level:28} {b['accuracy']:6.1%} -> "
              f"{a['accuracy']:6.1%}   on {b['n']}  "
              f"(chance {b['chance']:.2%})")

rise = log[-1]["control"] - log[0]["control"]
drop = log[0]["held_out"] - log[-1]["held_out"]
print(f"\nheld-out loss fell by {drop:.4f}")
print(f"control loss {'rose' if rise > 0 else 'fell'} by {abs(rise):.4f}")
if rise > 0.1:
    print("  the control corpus got worse. This is what catastrophic "
          "forgetting looks like before it reaches a benchmark.")
    if not REPLAY:
        print("  set REPLAY = 0.2 and run again: mixing general text back in "
              "is the ordinary remedy, and it was left off so that this run "
              "measured one thing.")

json.dump({"before": before, "after": after, "log": log},
          open(os.path.join(OUT, "result.json"), "w"), ensure_ascii=False, indent=2)
print(f"\nwrote {OUT}/result.json")

## 8. The other half, if you have it

Everything above is recall: every answer it asks for is stated in the corpus
the model just trained on.

The generalisation question is harder to ask now than it was, and that is
progress rather than a problem. The set that used to ask it is 1,134 Japanese
municipalities, and this corpus states the prefecture of 1,740 of them, so it
is no longer outside the corpus. What it still measures is whether the facts
survive being asked in a different form of words, by a different answer key,
outside the template the corpus used.

That answer key lives in `data/jp_municipalities.json` in the repository this
notebook came from. Run the cell and upload it, or skip it.

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    key = json.loads(next(iter(up.values())).decode("utf-8"))
except Exception as e:
    key = None
    print(f"no answer key ({e}); skipping the generalisation probe")

if key:
    random.Random(3).shuffle(key)
    sample = key[:PROBE_N]
    hits = 0
    for r in sample:
        q = OPEN.format(q=f"{r['ja']}は何県にありますか。")
        ids = tok(q, return_tensors="pt").to(device)
        with torch.no_grad():
            gen = model.generate(**ids, max_new_tokens=16, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        got = tok.decode(gen[0][ids["input_ids"].shape[1]:],
                         skip_special_tokens=True)
        hits += correct(got, r["pref_ja"])
    print(f"generalisation  {hits}/{len(sample)}  {hits / len(sample):.1%}  "
          f"(chance {1 / 47:.1%})")
    print("The corpus mentions one of these places. A number here that has "
          "not moved is the expected result, not a failure.")